# Student Activity: Outlier Detection Using Machine Learning

## Learning Objectives

By the end of this activity, you will be able to:

1. Apply **6+ different machine learning algorithms** for anomaly detection on non-time series data
2. Understand the **strengths and weaknesses** of different algorithm families
3. Implement and compare **distance-based, clustering-based, probabilistic, kernel-based, ensemble, and deep learning** methods
4. Evaluate and interpret results from **multiple anomaly detection approaches**
5. Choose the **most appropriate algorithm** for different data scenarios

---

## Introduction to Anomaly Detection with PyOD

Welcome to this hands-on activity on **anomaly detection** using the **Python Outlier Detection (PyOD)** library. You'll learn to detect anomalies in **non-time series data** using various machine learning approaches.

### What is Anomaly Detection?

**Anomaly detection** (also called outlier detection) is the process of identifying data points that deviate significantly from the majority of the data. Unlike statistical methods that rely on simple thresholds, machine learning approaches can:

- Handle **high-dimensional data** with multiple features
- Detect **complex patterns** that simple statistics might miss
- Learn **contextual relationships** between features
- Adapt to **non-linear patterns** in the data

### About PyOD

PyOD (Python Outlier Detection) is a comprehensive Python toolkit for detecting outliers in multivariate data. It provides:
- **30+ algorithms** from different families (statistical, proximity-based, clustering, neural networks)
- A **unified API** similar to scikit-learn
- **Scalable** implementations for large datasets
- Tools for **model evaluation and comparison**

### Important Note

PyOD algorithms work on **any tabular data**, not just time series! In this activity, we'll use **weight-height data** to demonstrate these techniques on cross-sectional data.

---

## Activity Structure

You will:
1. Load and explore the weight-height dataset (provided)
2. Implement algorithms from **6 different families** with guided TODOs
3. Compare results across different approaches
4. Complete a **quantitative evaluation** of algorithm performance
5. Answer reflection questions about method selection

Let's begin!

# Setup and Installation

First, let's install and verify all necessary libraries.

In [ ]:
!uv pip install pyod torch tqdm

In [ ]:
import matplotlib 
import pandas as pd
import pyod 
import numpy as np

print(f'''
Matplotlib -> {matplotlib.__version__}
pandas -> {pd.__version__}
PyOD -> {pyod.version.__version__}
''')

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams["figure.figsize"] = [12, 5]

# Dataset: Weight-Height Data

We'll use a weight-height dataset containing measurements of individuals. This is excellent for learning anomaly detection on non-time series data because:

- It has **clear expected relationships** (correlation between height and weight)
- It contains **potential outliers** (unusual measurements, data entry errors)
- It's a **real-world use case** (anomaly detection in health/biometric data)
- It demonstrates that PyOD works on **any tabular data**, not just time series

The dataset contains measurements for individuals, including gender, height (in inches), and weight (in pounds).

In [ ]:
file = Path("../data/weight-height.csv")
wh = pd.read_csv(file)
wh.head()

In [ ]:
# Basic data exploration
print(f"Dataset shape: {wh.shape}")
print(f"\nBasic statistics:\n{wh.describe()}")

In [ ]:
# Visualize the data
plt.figure(figsize=(10, 6))
plt.scatter(wh['Height'], wh['Weight'], alpha=0.5)
plt.xlabel('Height (inches)')
plt.ylabel('Weight (pounds)')
plt.title('Weight-Height Distribution')
plt.grid(True, alpha=0.3)
plt.show()

# Helper Functions

These functions will help you visualize outliers detected by different algorithms.

In [ ]:
def plot_outliers(data, predictions, algorithm_name, color='red'):
    """
    Visualize outliers detected by an algorithm.
    
    Parameters:
    -----------
    data : DataFrame
        The original dataset
    predictions : Series or array
        Binary predictions (0=normal, 1=outlier)
    algorithm_name : str
        Name of the algorithm for the title
    color : str
        Color for outlier points
    """
    outliers = data[predictions == 1]
    
    plt.figure(figsize=(10, 6))
    plt.scatter(data['Height'], data['Weight'], alpha=0.5, label='Normal', s=30)
    plt.scatter(outliers['Height'], outliers['Weight'], 
               color=color, s=100, alpha=0.7, label='Outliers', edgecolors='black')
    plt.xlabel('Height (inches)')
    plt.ylabel('Weight (pounds)')
    plt.title(f'{algorithm_name} Outlier Detection')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f"Number of outliers detected by {algorithm_name}: {len(outliers)}")
    return outliers

---

# Part 1: Distance-Based Algorithms

## Understanding Distance-Based Methods

Distance-based algorithms identify anomalies by measuring how **far** data points are from their neighbors. The intuition is simple:

- **Normal points** are surrounded by many neighbors (they're in dense regions)
- **Anomalous points** are isolated and far from other points (they're in sparse regions)

---

## Algorithm 1: K-Nearest Neighbors (KNN)

### How KNN Works for Anomaly Detection

The **KNN algorithm** identifies outliers based on the distance to their k-nearest neighbors:

1. **For each data point**, find its k closest neighbors
2. **Calculate a distance metric** (e.g., mean, median, or largest distance to the k neighbors)
3. **Assign an anomaly score** based on this distance
4. **Points with large distances** to their neighbors are considered outliers

**Key Hyperparameters:**
- `n_neighbors` (k): How many neighbors to consider (default: 5)
- `method`: How to aggregate distances ('mean', 'median', 'largest')
- `contamination`: Expected proportion of outliers in the data (e.g., 0.005 = 0.5%)

**When to use KNN:**
- ✅ Simple to understand and interpret
- ✅ Works well when outliers are far from normal points
- ❌ Computationally expensive for large datasets
- ❌ Sensitive to the choice of k

In [ ]:
from pyod.models.knn import KNN
from pyod.models.lof import LOF

### Your Task: Implement KNN Outlier Detection

Apply KNN to detect outliers in the **Height** feature only. We'll start with a simple univariate analysis.

In [ ]:
# TODO: Initialize a KNN detector with:
# - contamination=0.005 (expect 0.5% outliers)
# - method='mean' (use mean distance to neighbors)
# - n_neighbors=5 (consider 5 nearest neighbors)
knn = None  # Replace with your code

print(knn)

In [ ]:
# TODO: Fit the KNN model on the 'Height' feature
# Hint: Use wh[['Height']] to select the column as a DataFrame
# Your code here


In [ ]:
# TODO: Get predictions from the KNN model
# Hint: Use the .predict() method
knn_pred = None  # Replace with your code

# Convert to pandas Series for easier handling
knn_pred = pd.Series(knn_pred, index=wh.index)
print(f'Number of KNN outliers = {knn_pred.sum()}')

In [ ]:
# View the outliers detected by KNN
knn_outliers = wh[knn_pred == 1]
print("KNN Outliers:")
print(knn_outliers)

## Algorithm 2: Local Outlier Factor (LOF)

### How LOF Works for Anomaly Detection

**LOF** is an improvement over KNN that considers **local density** rather than just distance:

1. **Calculate local density** for each point based on distances to its k neighbors
2. **Compare each point's density** to the density of its neighbors
3. **Compute LOF score**: Ratio of the point's density to its neighbors' average density
4. **Points in low-density regions** relative to their neighbors are outliers

**Key Differences from KNN:**
- LOF considers **relative density** (compares local densities)
- KNN uses **absolute distance** (compares distances directly)
- LOF handles **varying density regions** better than KNN

**When to use LOF:**
- ✅ Data has **clusters with different densities**
- ✅ Need to detect **local** anomalies (outliers within a cluster)
- ✅ More robust than KNN for complex data distributions

### Your Task: Implement LOF and Compare with KNN

In [ ]:
# TODO: Initialize a LOF detector with:
# - contamination=0.005
# - n_neighbors=20 (LOF typically uses more neighbors than KNN)
lof = None  # Replace with your code

print(lof)

In [ ]:
# TODO: Fit the LOF model on the 'Height' feature
# Your code here


In [ ]:
# TODO: Get predictions from the LOF model
lof_pred = None  # Replace with your code

lof_pred = pd.Series(lof_pred, index=wh.index)
print(f'Number of LOF outliers = {lof_pred.sum()}')

In [ ]:
# View the outliers detected by LOF
lof_outliers = wh[lof_pred == 1]
print("LOF Outliers:")
print(lof_outliers)

### Exercise: Compare KNN vs LOF

Answer the following questions based on your results:

In [ ]:
# TODO: Calculate how many outliers are found by both algorithms
# Hint: Use set operations or boolean indexing
both_algorithms = None  # Replace with your code

# TODO: Calculate how many are unique to KNN
only_knn = None  # Replace with your code

# TODO: Calculate how many are unique to LOF
only_lof = None  # Replace with your code

print(f"Outliers detected by both: {both_algorithms}")
print(f"Outliers unique to KNN: {only_knn}")
print(f"Outliers unique to LOF: {only_lof}")

**Discussion Questions:**
1. Which algorithm found more outliers?
2. Why might the algorithms disagree on some points?
3. Which algorithm would you trust more for this data? Why?

---

# Part 2: Clustering-Based Algorithms

## Understanding Clustering-Based Methods

Clustering-based algorithms use a different intuition:

**Core Idea**: 
- First, **group similar data points** into clusters
- Then, identify points that **don't fit well** into any cluster
- Points far from cluster centers or in small/sparse clusters are anomalies

**Advantage over distance-based methods**:
- Can handle **complex data distributions** with multiple modes
- More efficient for **large datasets** (cluster first, then evaluate)
- Can detect anomalies as **points between clusters**

---

## Algorithm: Cluster-Based Local Outlier Factor (CBLOF)

### How CBLOF Works

CBLOF combines clustering with outlier scoring:

1. **Clustering Phase**: 
   - Apply clustering (e.g., K-means) to partition the data into `n_clusters` groups
   - Classify clusters as **large** (many points) or **small** (few points)

2. **Outlier Scoring**:
   - For points in **large clusters**: Score based on distance to cluster center
   - For points in **small clusters**: Score based on distance to nearest large cluster
   - Intuition: Small clusters are likely to contain outliers

**Key Hyperparameters:**
- `n_clusters`: Number of clusters to create
- `contamination`: Expected proportion of outliers
- `alpha`: Threshold to classify clusters as large vs. small (default: 0.9)
- `beta`: Penalty multiplier for small clusters (default: 5)

**When to Use CBLOF:**
- ✅ Data has **natural groupings** or clusters
- ✅ Need better **scalability** than pure distance-based methods
- ❌ Need to choose the number of clusters (requires domain knowledge)

In [ ]:
from pyod.models.cblof import CBLOF

### Your Task: Implement CBLOF

In [ ]:
# TODO: Initialize a CBLOF detector with:
# - n_clusters=8 (try to find 8 natural groupings)
# - contamination=0.001 (expect 0.1% outliers)
# - alpha=0.9, beta=5 (default values)
cblof = None  # Replace with your code

print(cblof)

In [ ]:
# TODO: Fit CBLOF on the 'Height' feature and get predictions
# Your code here

cblof_pred = None  # Replace with your code

print(f'Number of CBLOF outliers = {cblof_pred.sum()}')

In [ ]:
# View CBLOF outliers
cblof_outliers = wh[cblof_pred == 1]
print("CBLOF Outliers:")
print(cblof_outliers)

---

# Part 3: Probabilistic Algorithms

## Understanding Probabilistic Methods

Unlike distance or clustering methods, probabilistic algorithms model the **probability distribution** of the data:

**Core Idea**:
- Learn the **joint probability distribution** of all features
- Calculate how **probable** each data point is under this distribution
- Points with **low probability** are considered anomalies

**Advantages**:
- ✅ Theoretically grounded in **statistics and probability theory**
- ✅ Can handle **dependencies between features** 
- ✅ Often **parameter-free** or require minimal tuning
- ✅ Fast prediction after training

---

## Algorithm 1: COPOD (Copula-Based Outlier Detection)

### How COPOD Works

COPOD uses **copula theory** from statistics to model dependencies:

1. **Marginal Distributions**: Model each feature's distribution independently
2. **Tail Probabilities**: Compute left-tail and right-tail probabilities for each feature
3. **Copula-Based Combination**: Use copula functions to combine probabilities across features
4. **Outlier Score**: Lower probability = higher anomaly score

**Key Advantages:**
- **Parameter-free**: No hyperparameters to tune (except contamination)
- **Fast**: Linear time complexity O(n)
- **Distribution-free**: Doesn't assume Gaussian or any specific distribution

**When to Use COPOD:**
- ✅ Need a **simple, fast** algorithm with **no tuning**
- ✅ Data has **multiple features** with potential dependencies
- ✅ Working with **large datasets** (very efficient)

In [ ]:
from pyod.models.copod import COPOD
from pyod.models.ecod import ECOD

### Your Task: Implement COPOD

Now we'll use **both Height and Weight** features together!

In [ ]:
# TODO: Initialize COPOD with contamination=0.005
copod = None  # Replace with your code

# TODO: Fit COPOD on both 'Height' and 'Weight' features
# Hint: Use wh[['Height', 'Weight']]
# Your code here


In [ ]:
# TODO: Get predictions from COPOD
copod_pred = None  # Replace with your code

copod_pred = pd.Series(copod_pred, index=wh.index)
print(f'Number of COPOD outliers = {copod_pred.sum()}')

In [ ]:
# TODO: Visualize COPOD outliers using the helper function
copod_outliers = plot_outliers(wh, copod_pred, 'COPOD', color='orange')

## Algorithm 2: ECOD (Empirical Cumulative Distribution Outlier Detection)

ECOD is a simpler, even faster variant that assumes feature independence.

### Your Task: Implement ECOD and Compare with COPOD

In [ ]:
# TODO: Initialize and fit ECOD (similar to COPOD)
ecod = None  # Replace with your code

# Your code here

# TODO: Get predictions
ecod_pred = None  # Replace with your code

ecod_pred = pd.Series(ecod_pred, index=wh.index)
print(f'Number of ECOD outliers = {ecod_pred.sum()}')

In [ ]:
# TODO: Visualize ECOD outliers
ecod_outliers = plot_outliers(wh, ecod_pred, 'ECOD', color='green')

---

# Part 4: Kernel-Based Algorithms

## Understanding Kernel-Based Methods

Kernel-based algorithms use **kernel functions** to transform data into higher-dimensional spaces where separation between normal and anomalous points becomes easier.

**Core Concept**:
- Data that's **not linearly separable** in the original space...
- ...might be **separable in a transformed (kernel) space**
- Use the **"kernel trick"** to work in high dimensions without explicitly computing the transformation

---

## Algorithm: One-Class SVM (OCSVM)

### How One-Class SVM Works

Unlike traditional SVM (which learns a boundary between two classes), **One-Class SVM** learns a boundary around **one class** (normal data):

1. **Map to Feature Space**: Use a kernel function to implicitly transform data
2. **Learn the Decision Boundary**: Fit a hyperplane that **encloses most normal points**
3. **Classify New Points**: Points inside → normal, points outside → anomalies

### Key Hyperparameters

- **`kernel`**: The kernel function ('rbf', 'linear', 'poly', 'sigmoid')
  - **'rbf'**: Most popular, handles non-linear patterns
- **`gamma`**: Kernel coefficient ('auto' or float)
- **`nu`**: Upper bound on fraction of outliers (typically 0.5)

### The Importance of Scaling

⚠️ **Critical**: OCSVM is **sensitive to feature scales**!
- Features with large ranges dominate the distance calculations
- **Always standardize/normalize** features before applying OCSVM
- Use `StandardScaler` or PyOD's `standardizer` utility

**When to Use OCSVM:**
- ✅ Data has **complex, non-linear patterns**
- ✅ Need a **robust** decision boundary
- ✅ Can afford to **scale your features** properly
- ❌ Have very large datasets (can be slow)

In [ ]:
from pyod.models.ocsvm import OCSVM
from pyod.utils.utility import standardizer

### Your Task: Implement OCSVM Without Scaling (to see why scaling matters)

In [ ]:
# TODO: Initialize OCSVM with:
# - contamination=0.005
# - kernel='rbf'
# - gamma='auto'
# - nu=0.5
ocsvm_unscaled = None  # Replace with your code

# TODO: Fit on unscaled data (Height and Weight)
# Your code here

# TODO: Get predictions
ocsvm_unscaled_pred = None  # Replace with your code

ocsvm_unscaled_pred = pd.Series(ocsvm_unscaled_pred, index=wh.index)
print(f'Number of OCSVM outliers (unscaled) = {ocsvm_unscaled_pred.sum()}')

In [ ]:
# Visualize unscaled results
ocsvm_unscaled_outliers = plot_outliers(wh, ocsvm_unscaled_pred, 'OCSVM (Unscaled)', color='purple')

### Your Task: Implement OCSVM WITH Scaling

Now let's see how scaling improves the results!

In [ ]:
# TODO: Scale the data using standardizer
scaled_data = None  # Replace with your code using standardizer()

# TODO: Initialize a new OCSVM model (same parameters as before)
ocsvm_scaled = None  # Replace with your code

# TODO: Fit on scaled data
# Your code here

# TODO: Get predictions
ocsvm_scaled_pred = None  # Replace with your code

ocsvm_scaled_pred = pd.Series(ocsvm_scaled_pred, index=wh.index)
print(f'Number of OCSVM outliers (scaled) = {ocsvm_scaled_pred.sum()}')

In [ ]:
# Visualize scaled results
ocsvm_scaled_outliers = plot_outliers(wh, ocsvm_scaled_pred, 'OCSVM (Scaled)', color='magenta')

### Exercise: Compare Scaled vs Unscaled Results

Answer:
1. How many outliers were detected without scaling vs with scaling?
2. Why does scaling make such a difference for OCSVM?
3. Which result looks more reasonable when you visualize it?

---

# Part 5: Ensemble Methods

## Understanding Ensemble Methods

**Ensemble methods** combine multiple models or decision strategies to improve overall performance:

**Core Principle**: "Wisdom of the crowd"
- Individual models might make mistakes
- But **combining multiple models** often produces better results
- Different models capture different aspects of anomalies

---

## Algorithm: Isolation Forest (IForest)

### How Isolation Forest Works

IForest uses a unique approach based on **isolation** rather than distance or density:

**Key Insight**: 
- **Anomalies are rare and different** → easier to isolate
- **Normal points are common and similar** → harder to isolate

### The Algorithm

1. **Build Isolation Trees**: Randomly split data recursively
2. **Measure Path Length**: Count how many splits needed to isolate each point
3. **Aggregate Across Trees**: Average path length across all trees
   - **Anomalies** → isolated quickly → **short paths**
   - **Normal points** → many splits needed → **long paths**

### Key Hyperparameters

- **`n_estimators`**: Number of isolation trees (default: 100)
- **`contamination`**: Expected proportion of outliers
- **`bootstrap`**: Whether to use sampling with replacement

### Why Isolation Forest is Popular

- ✅ **Fast**: Linear time complexity O(n)
- ✅ **Scalable**: Works well with large datasets
- ✅ **Few parameters**: Easy to use
- ✅ **Handles high-dimensional data** well
- ✅ **No need for scaling** (unlike OCSVM)

**When to Use IForest:**
- ✅ **Large datasets** (very efficient)
- ✅ **High-dimensional data**
- ✅ Need **fast training and prediction**
- ✅ Want a **parameter-free** method (works well with defaults)

In [ ]:
from pyod.models.iforest import IForest

### Your Task: Implement Isolation Forest

In [ ]:
# TODO: Initialize IForest with:
# - contamination=0.005
# - n_estimators=100
# - bootstrap=False
# - random_state=42 (for reproducibility)
iforest = None  # Replace with your code

# TODO: Fit on Height and Weight (NO scaling needed!)
# Your code here


In [ ]:
# TODO: Get predictions from IForest
iforest_pred = None  # Replace with your code

iforest_pred = pd.Series(iforest_pred, index=wh.index)
print(f'Number of IForest outliers = {iforest_pred.sum()}')

In [ ]:
# TODO: Visualize IForest outliers
iforest_outliers = plot_outliers(wh, iforest_pred, 'Isolation Forest', color='cyan')

### Exercise: Why doesn't IForest need scaling?

Think about:
- How IForest makes splits (random feature selection)
- Why distance-based methods (KNN, OCSVM) are sensitive to scale
- What this means for practical applications

---

# Part 6: Deep Learning Methods

## Understanding Deep Learning-Based Methods

Deep learning brings **neural networks** to anomaly detection:

**Core Idea**:
- Use neural networks to **learn compressed representations** of normal data
- Points that **reconstruct poorly** from these representations are anomalies
- The network learns what "normal" looks like automatically

**Advantages**:
- ✅ **Automatic feature learning**: No manual feature engineering needed
- ✅ **Handle complex patterns**: Can model highly non-linear relationships
- ✅ **Scalable**: Works with high-dimensional data

**Challenges**:
- ❌ **Computationally expensive**: Requires more time and resources
- ❌ **Hyperparameter tuning**: Learning rate, epochs, batch size
- ❌ **Less interpretable**: "Black box" compared to simpler methods

---

## Algorithm: AutoEncoder

### What is an AutoEncoder?

An **AutoEncoder** is a neural network trained to **reconstruct its input**:

```
Input → Encoder → Compressed Representation → Decoder → Reconstructed Output
```

### How AutoEncoders Detect Anomalies

1. **Training Phase** (on normal data):
   - **Encoder**: Compresses input to low-dimensional representation
   - **Decoder**: Reconstructs input from the compressed representation
   - **Objective**: Minimize reconstruction error for normal points

2. **Detection Phase**:
   - Calculate **reconstruction error** = |input - reconstructed output|
   - **Normal points**: Small reconstruction error
   - **Anomalous points**: Large reconstruction error

### Key Hyperparameters

- **`lr` (learning rate)**: How fast the network learns (typical: 0.001 - 0.01)
- **`epoch_num`**: Number of complete passes through the data (start with 10-50)
- **`batch_size`**: Number of samples per training update (typical: 16, 32, 64)

**When to Use AutoEncoder:**
- ✅ Have **sufficient data** to train a neural network
- ✅ Data has **complex, non-linear patterns**
- ✅ Can afford the **computational cost**

In [ ]:
from pyod.models.auto_encoder import AutoEncoder

### Your Task: Implement AutoEncoder

Start with a small number of epochs to see how it performs.

In [ ]:
# TODO: Initialize AutoEncoder with:
# - contamination=0.005
# - lr=0.001 (learning rate)
# - epoch_num=10 (start small)
# - batch_size=32
auto_encoder = None  # Replace with your code

# TODO: Fit on Height and Weight
# Note: This will take longer than previous algorithms!
# Your code here


In [ ]:
# TODO: Get predictions
ae_pred = None  # Replace with your code

ae_pred = pd.Series(ae_pred, index=wh.index)
print(f'Number of AutoEncoder outliers (10 epochs) = {ae_pred.sum()}')

In [ ]:
# TODO: Visualize AutoEncoder outliers
ae_outliers = plot_outliers(wh, ae_pred, 'AutoEncoder (10 epochs)', color='brown')

### Your Task: Experiment with More Epochs

Does training longer improve the results?

In [ ]:
%%time
# TODO: Train AutoEncoder with 50 epochs instead of 10
# Initialize, fit, and predict
auto_encoder_50 = None  # Replace with your code

# Your code here

ae_pred_50 = None  # Replace with your code

ae_pred_50 = pd.Series(ae_pred_50, index=wh.index)
print(f'Number of AutoEncoder outliers (50 epochs) = {ae_pred_50.sum()}')

In [ ]:
# Visualize 50-epoch results
ae_outliers_50 = plot_outliers(wh, ae_pred_50, 'AutoEncoder (50 epochs)', color='red')

### Exercise: Analyze Training Time vs Performance

Consider:
1. How much longer did 50 epochs take compared to 10?
2. Did the results improve significantly?
3. Would you use more or fewer epochs for this dataset? Why?

---

# Part 7: Quantitative Evaluation and Comparison

## Understanding Algorithm Agreement

Now that you've implemented multiple algorithms, let's quantitatively compare their results. Different algorithms may find different anomalies because they use different definitions of "unusual."

This section will help you:
- Calculate **overlap between algorithms**
- Identify **consensus outliers** (detected by many algorithms)
- Find **algorithm-specific outliers** (detected by only one method)
- Make **informed decisions** about which results to trust

### Your Task: Create a Comparison DataFrame

Combine all your predictions into a single DataFrame for easy comparison.

In [ ]:
# TODO: Create a DataFrame with all predictions
# Include: KNN, LOF, CBLOF, COPOD, ECOD, OCSVM (scaled), IForest, AutoEncoder

comparison_df = pd.DataFrame({
    'KNN': knn_pred,
    'LOF': lof_pred,
    # TODO: Add the rest of your predictions
    # 'CBLOF': ...,
    # 'COPOD': ...,
    # 'ECOD': ...,
    # 'OCSVM_scaled': ...,
    # 'IForest': ...,
    # 'AutoEncoder': ...
}, index=wh.index)

comparison_df.head()

### Your Task: Calculate Algorithm Overlap

In [ ]:
# TODO: Calculate how many algorithms flagged each point as an outlier
# Hint: Sum across columns for each row
comparison_df['num_algorithms'] = None  # Replace with your code

# TODO: Display points detected by 3 or more algorithms (high confidence outliers)
high_confidence_outliers = None  # Replace with your code

print(f"Number of high-confidence outliers (3+ algorithms): {len(high_confidence_outliers)}")
print("\nHigh-confidence outliers:")
print(high_confidence_outliers)

In [ ]:
# TODO: Visualize the high-confidence outliers on the original data
plt.figure(figsize=(10, 6))
plt.scatter(wh['Height'], wh['Weight'], alpha=0.5, label='Normal', s=30)
# TODO: Add scatter plot for high-confidence outliers
# Your code here

plt.xlabel('Height (inches)')
plt.ylabel('Weight (pounds)')
plt.title('High-Confidence Outliers (Detected by 3+ Algorithms)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Your Task: Create a Confusion Matrix Between Algorithms

How much do different algorithms agree with each other?

In [ ]:
# TODO: Calculate pairwise agreement between algorithms
# For each pair of algorithms, count how many outliers they agree on

algorithms = comparison_df.columns[:-1]  # Exclude 'num_algorithms'
agreement_matrix = pd.DataFrame(index=algorithms, columns=algorithms)

for algo1 in algorithms:
    for algo2 in algorithms:
        # TODO: Calculate how many points both algorithms marked as outliers
        # Hint: Use & for logical AND
        agreement = None  # Replace with your code
        agreement_matrix.loc[algo1, algo2] = agreement

# Convert to numeric
agreement_matrix = agreement_matrix.astype(int)
print("Algorithm Agreement Matrix (number of shared outliers):")
print(agreement_matrix)

In [ ]:
# Visualize the agreement matrix as a heatmap
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(agreement_matrix, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Algorithm Agreement Matrix\n(Number of Shared Outlier Detections)')
plt.tight_layout()
plt.show()

### Your Task: Identify Algorithm-Specific Outliers

In [ ]:
# TODO: For each algorithm, find outliers that ONLY it detected
# (i.e., num_algorithms == 1 for that specific algorithm)

for algo in algorithms:
    # TODO: Find points where this algorithm = 1 AND num_algorithms = 1
    unique_outliers = None  # Replace with your code
    
    print(f"\n{algo} unique outliers: {len(unique_outliers)}")
    if len(unique_outliers) > 0:
        print(unique_outliers[['Height', 'Weight']])

### Exercise: Interpretation Questions

Based on your quantitative analysis, answer:

1. **Which two algorithms agree the most?** Why might this be?

2. **Which algorithm found the most unique outliers?** What does this tell you about that algorithm?

3. **Do you trust the high-confidence outliers more than algorithm-specific ones?** Why or why not?

4. **If you had to pick ONE algorithm for production use on this data, which would you choose?** Justify your answer.

---

# Part 8: Challenge Exercise

## Apply Multiple Algorithms and Compare Results

Now it's your turn to be the data scientist! Complete this challenge:

### The Challenge

1. Choose **3 algorithms** from different families that you think would work best for this data
2. Tune their hyperparameters (if applicable) to improve performance
3. Compare the results quantitatively
4. Write a brief recommendation on which algorithm to use and why

### Your Task

In [ ]:
# TODO: Choose and implement 3 algorithms
# Algorithm 1: [Your choice]
algo1 = None  # Your code here

# Algorithm 2: [Your choice]
algo2 = None  # Your code here

# Algorithm 3: [Your choice]
algo3 = None  # Your code here

In [ ]:
# TODO: Compare their results
# - How many outliers did each find?
# - How much do they overlap?
# - Which one seems most reasonable based on visualization?

# Your analysis here

### TODO: Write Your Recommendation

**Based on your analysis, which algorithm would you recommend for this weight-height dataset?**

Consider:
- **Accuracy**: Does it find reasonable outliers?
- **Speed**: How fast does it run?
- **Interpretability**: Can you explain the results to stakeholders?
- **Robustness**: Does it require careful tuning?

Your recommendation:
[Write your answer here]

---

# Part 9: Reflection Questions

## Understanding When to Use Each Method

Answer the following questions based on your experience in this activity:

### Question 1: Distance-Based vs Ensemble Methods

**When would you choose distance-based methods (KNN, LOF) over ensemble methods (IForest)?**

Consider:
- Dataset size
- Interpretability needs
- Computational resources
- Nature of anomalies

*Your answer:*

---

### Question 2: The Importance of Scaling

**Why does scaling matter for OCSVM but not for IForest?**

Think about:
- How each algorithm makes decisions
- The role of distance calculations
- Random splitting vs distance measurements

*Your answer:*

---

### Question 3: Interpretability vs Performance Trade-offs

**What are the trade-offs between interpretability and performance in anomaly detection?**

Compare:
- Simple methods (KNN, COPOD)
- Complex methods (AutoEncoder, OCSVM)

*Your answer:*

---

### Question 4: Real-World Application

**If you were deploying an anomaly detection system for healthcare data (similar to our weight-height data but with more features), which algorithm would you choose and why?**

Consider:
- False positive cost (flagging healthy patients)
- False negative cost (missing unhealthy patients)
- Explainability to doctors
- Real-time performance needs

*Your answer:*

---

### Question 5: Multiple Algorithm Approach

**Based on your quantitative evaluation, do you think using multiple algorithms together (ensemble of ensembles) would be beneficial? Why or why not?**

*Your answer:*

---

# Summary: What You've Learned

Congratulations! You've completed a comprehensive activity on ML-based anomaly detection. You've:

## Algorithms Implemented

1. **Distance-Based**: KNN, LOF
2. **Clustering-Based**: CBLOF
3. **Probabilistic**: COPOD, ECOD
4. **Kernel-Based**: OCSVM (with scaling comparison)
5. **Ensemble**: Isolation Forest
6. **Deep Learning**: AutoEncoder (with epoch tuning)

## Key Insights

- **Different algorithms find different outliers** because they use different definitions of "unusual"
- **Feature scaling is critical** for some algorithms (OCSVM) but not others (IForest)
- **No single "best" algorithm** exists - the choice depends on your data and requirements
- **Consensus outliers** (detected by multiple algorithms) are often more trustworthy
- **Parameter-free methods** (COPOD, ECOD) are great starting points
- **Deep learning** offers power but requires more data and tuning

## Decision Framework

| Scenario | Recommended Algorithm |
|----------|----------------------|
| **Large dataset, need speed** | Isolation Forest, COPOD, ECOD |
| **Small dataset** | KNN, LOF, OCSVM |
| **High-dimensional data** | Isolation Forest, AutoEncoder |
| **Complex non-linear patterns** | OCSVM (RBF), AutoEncoder |
| **Varying density clusters** | LOF, CBLOF |
| **Need interpretability** | KNN, Isolation Forest |
| **Parameter-free method** | COPOD, ECOD |

## PyOD Workflow (Remember This!)

```python
# 1. Initialize
model = Algorithm(contamination=0.005, ...)

# 2. Fit
model.fit(X)

# 3. Predict
predictions = model.predict(X)  # 0=normal, 1=outlier

# 4. Get scores (optional)
scores = model.decision_scores_  # Higher = more anomalous
```

---

## Next Steps

- Try these algorithms on your own datasets
- Explore more PyOD algorithms (30+ available!)
- Learn about time-series specific anomaly detection
- Study advanced ensemble methods (combining multiple algorithms)
- Investigate domain-specific anomaly detection techniques

---

**Great job completing this activity!**